<a href="https://colab.research.google.com/github/bitsteller/dataanalys-for-kollektivtrafik/blob/main/Kod/Laboration%203/Uppgift2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Laboration 3 – Uppgift 2

Den här uppgiften handlar om att bearbeta data om tåg som har passerat Hässleholm med **Polars**.

## Jobba så här
1. Kör exempelcellen i varje avsnitt.
2. Fyll i kod i övningscellen under.
3. Kör övningscellen och kontrollera att resultatet ser rimligt ut.

## Om datasetet

Datasetet innehåller information alla genomgående tåg (som har både ankomst och avgång) som har passerat Hässleholm under 2025.
Mer detaljerad information om datasetet finns i labbinstruktionen.


## 1) Läsa in data med polars

Vi börjar med att läsa in `trains.parquet` och kontrollera struktur och datatyper.
En parquet fil innehåller data i tabellformat (ungefär som en Excel eller CSV fil).

* `read_parquet()` läser in en parquet-fil som en tabell (så kallad DataFrame)
* `.height` anger antalet rader
* `.columns` listar namnen på alla kolumner

In [ ]:
import polars as pl

BASE_URL = "https://raw.githubusercontent.com/bitsteller/dataanalys-for-kollektivtrafik/main/Data/Laboration%203"
TRAINS_URL = f"{BASE_URL}/trains.parquet"

trains = pl.read_parquet(TRAINS_URL)
print("Antal rader i trains:", trains.height)
print("Kolumner:", trains.columns)


Antal rader i trains: 36670
Kolumner: ['TrainRunID', 'Date', 'Weekday', 'Hour', 'TimeOfDay', 'TrainOwner', 'ProductName', 'TrainLength', 'FromLocationSignature', 'ToLocationSignature', 'TrackAtLocation', 'DeviationCodes', 'PlannedDwellTime', 'ActualDwellTime']


In [3]:
# Övning: Läs in locations.parquet som finns på URL:n `LOCATIONS_URL` och skriv ut antal rader + kolumnnamn.
LOCATIONS_URL = f"{BASE_URL}/locations.parquet"

locations = None  # byt ut None mot ett kommando som läser in locations
print("Antal rader i locations:", None)  # byt ut None
print("Kolumner:", None)  # byt ut None


Antal rader i locations: None
Kolumner: None


## 2) Inspektera datasetet

Ett enkelt sätt att inspektera tabellen är att titta på de första raderna med funktionen `head()`.

* `head()` visar de första raderna i en tabell (standard är 10 men det går att ange ett annat antal som parameter)
* `select()` kan användas för att behålla endast vissa kolumner

In [ ]:
trains.head()

Head (trains):


TrainRunID,Date,Weekday,Hour,TimeOfDay,TrainOwner,ProductName,TrainLength,FromLocationSignature,ToLocationSignature,TrackAtLocation,DeviationCodes,PlannedDwellTime,ActualDwellTime
i64,date,str,i8,str,str,str,str,str,str,str,list[str],f64,f64
225590407907666,2024-12-15,"""Sun""",0,"""night (22-5)""","""Ö-TÅG""","""Öresundståg""","""""","""Dk.kh""","""Cr""","""3""",null,2.0,3.116667
5157817509284191,2024-12-15,"""Sun""",0,"""night (22-5)""","""SKANE""","""Pågatågen""","""""","""Cr""","""Hie""","""2""",null,5.0,7.516667
446724905912515,2024-12-15,"""Sun""",5,"""morning (5-9)""","""Ö-TÅG""","""Öresundståg""","""""","""Cr""","""Dk.kh""","""1b""",null,2.0,1.866667
3415546339555511,2024-12-15,"""Sun""",6,"""morning (5-9)""","""SKANE""","""Pågatågen""","""""","""Cr""","""Trg""","""2""",null,2.0,3.4
3909395287938004,2024-12-15,"""Sun""",6,"""morning (5-9)""","""Ö-TÅG""","""Öresundståg""","""""","""Cr""","""Dk.kh""","""1b""",null,2.0,6.233333


In [ ]:
# Övning: Visa de första raderna på `trains_selected` som endast innehåller kolumnerna `Date`, `Weekday` och `TrainOwner`.
trains_selected = trains.select(["Date", "Weekday", "TrainOwner"])  # byt gärna kolumner

#Använd funktionen head() här


shape: (3, 3)
┌────────────┬─────────┬────────────┐
│ Date       ┆ Weekday ┆ TrainOwner │
│ ---        ┆ ---     ┆ ---        │
│ date       ┆ str     ┆ str        │
╞════════════╪═════════╪════════════╡
│ 2024-12-15 ┆ Sun     ┆ Ö-TÅG      │
│ 2024-12-15 ┆ Sun     ┆ SKANE      │
│ 2024-12-15 ┆ Sun     ┆ Ö-TÅG      │
└────────────┴─────────┴────────────┘


In [20]:
# Bonusövning: Funktionen describe() kan användas på en tabell för få sammanfattande statistik som medelvärde och percentiler
# Testa lägga till fler numeriska kolumner (Exempel: "Hour", "PlannedDwellTime", "ActualDwellTime")

trains.select(["Hour"]).describe()

statistic,Hour
str,f64
"""count""",36670.0
"""null_count""",0.0
"""mean""",13.728334
"""std""",5.386701
"""min""",0.0
"""25%""",9.0
"""50%""",14.0
"""75%""",18.0
"""max""",23.0


## 3) Filtrera data

Nu filtrerar vi fram en delmängd för att räkna ut hur många tåg som passerar i snitt på en vardag.

* `filter()` används för att filtrerar raderna där ett viist kriterium är sant
* `pl.col()` används för att referera till en kolumn
* `is_in()` är sant om värdet för en kolumn finns i den angivna listan, annars false
* `&` används för att kombinera kriterer där alla måste vara sanna för att raden ska vara kvar
* `==`, `<`, `>=`, osv. kan användas för jämförelser

In [25]:
vardag = trains.filter(
    (pl.col("Weekday").is_in(["Mon", "Tue", "Wed", "Thu", "Fri"]))
)

print("Antal tåg vardagar:", vardag.height)


Antal tåg vardagar: 28497


In [26]:
# Övning: Beräkna antal tåg som har gått på helger (lör, sön)





In [28]:
# Övning: Beräkna antal tåg som har gått mellan klockan 6 och 9 under vardagar genom att kombinera flera kriterier.

morgon_vardag = trains.filter(
    (pl.col("Weekday").is_in(["Mon", "Tue", "Wed", "Thu", "Fri"]))
    & True #ersätt True med ett kriterium som filtrerar tåg som går klockan 6 eller senare
    & (pl.col("Hour") < 9)
)

print("Rader morgon vardag:", morgon_vardag.height)

Rader morgon vardag: 6428


## 4) Gruppera data

Vi kan också grupperar för att se antalet tåg per vardag.

* `group_by()` grupperar alla rader med samma värde i den angivna kolumnen
* `agg()` används för att räkna ut mått för varje grupp
* `len()` räknar ut antalet per grupp
* `alias()` ändrar namnet på en kolumn
* `sort()` sorterar tabellen enligt en eller flera kolumner


In [42]:
per_veckodag = (
    trains.group_by("Weekday")  
    .agg(pl.len().alias("Antal"))
    .sort("Antal", descending=True)
)

per_veckodag


Weekday,Antal
str,u32
"""Thu""",6308
"""Mon""",6174
"""Fri""",5666
"""Tue""",5401
"""Wed""",4948
"""Sun""",4436
"""Sat""",3737


In [ ]:
# Övning: Gruppera på ProductName för att se antalet tåg per tågtyp

resor_per_dag = (
    trains.group_by("ProductName") # testa gärna annan gruppering
    #använd agg() för att räkna ut antal per grupp
)

resor_per_dag


## 5) Aggregation

Nu beräknar vi flera mått samtidigt för att förstå uppehållstider.

In [ ]:
agg_per_dag = (
    trains.group_by("Date")
    .agg([
        pl.len().alias("AntalTag"),
        pl.col("ActualDwellTime").mean().alias("MeanActualDwell"),
        pl.col("ActualDwellTime").max().alias("MaxActualDwell"),
    ])
    .sort("Date")
)

print(agg_per_dag.head(10))


In [ ]:
# Övning: Lägg till en aggregation för minsta ActualDwellTime per Date.

ovning_agg = (
    trains.group_by("Date")
    .agg([
        pl.len().alias("AntalTag"),
        pl.col("ActualDwellTime").min().alias("MinActualDwell"),
    ])
    .sort("Date")
)

print(ovning_agg.head(10))


## Bonus) Sammanfoga tabeller

Vi kopplar tågrörelser till stationsnamn via locations-tabellen.

In [ ]:
locations = pl.read_parquet(LOCATIONS_URL)

joined = trains.join(
    locations.select(["LocationSignature", "AdvertisedLocationName"]),
    left_on="FromLocationSignature",
    right_on="LocationSignature",
    how="left",
)

print("Rader trains:", trains.height)
print("Rader efter join:", joined.height)
print(joined.select(["FromLocationSignature", "AdvertisedLocationName"]).head(10))


In [ ]:
# Övning: Gör en join för ToLocationSignature och visa 10 rader med destinationens namn.

joined_to = trains.join(
    locations.select(["LocationSignature", "AdvertisedLocationName"]),
    left_on="ToLocationSignature",
    right_on="LocationSignature",
    how="left",
)

print(joined_to.select(["ToLocationSignature", "AdvertisedLocationName"]).head(10))


## Reflektion

1. Vilken veckodag verkar ha flest observationer i datan?
2. Hur skiljer sig genomsnittlig `ActualDwellTime` mellan olika dagar?
3. Vad händer med datan om du byter `how="left"` till `how="inner"` i bonus-joinen?
